# Construyendo aplicaciones de IA con Gradio y NVIDIA NIM

En los notebooks anteriores interactuamos con LLMs directamente desde Python.
En este notebook construiremos interfaces web interactivas usando **Gradio**,
conectadas a **NVIDIA NIM** y al modelo **Nemotron 3 Super**.

Gradio permite crear interfaces web para modelos de IA con muy poco código. Es
muy útil para demostraciones, prototipos y herramientas internas.

## Prerrequisitos

Configura `NVIDIA_API_KEY` en los secretos de Google Colab o en el archivo `.env` si trabajas localmente. Si todavía no la tienes, revisa las instrucciones del notebook 03. No incluyas la clave en el notebook ni la compartas.

In [ ]:
# %%bash
!pip install -q --upgrade gradio openai python-dotenv

## 1. Configuración del Entorno

In [1]:
import os
import gradio as gr
from openai import OpenAI
from dotenv import load_dotenv

In [ ]:
# Cargar la clave de NVIDIA desde secretos de Colab o desde .env / variables de entorno.
load_dotenv()
NVIDIA_API_KEY = os.getenv("NVIDIA_API_KEY")
if not NVIDIA_API_KEY:
    try:
        from google.colab import userdata
        NVIDIA_API_KEY = userdata.get("NVIDIA_API_KEY")
    except (ImportError, KeyError, ValueError):
        pass

if NVIDIA_API_KEY:
    print("NVIDIA API Key cargada correctamente")
else:
    raise RuntimeError("Falta NVIDIA_API_KEY. Configúrala en secretos de Colab o en .env antes de continuar.")

In [3]:
# Cliente compatible con OpenAI conectado a NVIDIA NIM
client = OpenAI(
    base_url="https://integrate.api.nvidia.com/v1",
    api_key=NVIDIA_API_KEY,
)

In [4]:
# Modelo usado en todo el notebook
MODELO = "nvidia/nemotron-3-super-120b-a12b"

## 2. Tu primera interfaz con Gradio

Antes de conectar el LLM, entendamos cómo funciona Gradio con un ejemplo simple.

`gr.Interface` toma una función de Python y la convierte en una interfaz web:

- `fn`: función que se ejecutará.
- `inputs`: componentes de entrada.
- `outputs`: componentes de salida.

In [5]:
# Función simple que devuelve lo que el usuario escribe
def echo(message):
    return f"Dijiste: {message}"

In [6]:
# Creamos la interfaz — Gradio levanta un servidor web local automáticamente
# Al ejecutar esta celda verás un link para abrir la interfaz en el navegador
demo = gr.Interface(
    fn=echo,
    inputs="textbox",
    outputs="textbox",
    title="Echo Bot",
    description="Una interfaz simple que repite lo que escribes",
    flagging_mode="never"  # desactiva el botón de reportar respuestas
)

In [7]:
demo.launch(
    server_name="0.0.0.0",
    server_port=8081,
    show_error=True
)

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://9e0f160801abb7613c.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [8]:
demo.close()

Closing server running on port: 8081


### Análisis — Echo Bot

Esta primera interfaz utiliza `gr.Interface` para relacionar una función de Python (`echo`) con una caja de entrada y una caja de salida. No utiliza un modelo de inteligencia artificial ni consume tokens de NVIDIA. Su propósito es verificar el funcionamiento básico de Gradio y el lanzamiento de interfaces. La celda `demo.close()` permite cerrar el servidor antes de abrir otro en el mismo puerto.

**Prueba sugerida:** escribir «Hola Gradio» y verificar que la salida sea «Dijiste: Hola Gradio».

## 3. Chatbot básico con NVIDIA NIM

Ahora conectamos Gradio con NVIDIA NIM. La función `get_response` llama a la API
y devuelve el texto; Gradio se encarga de presentarlo en la interfaz.

In [9]:
def get_response(message: str) -> str:
    """Obtiene una respuesta de NVIDIA NIM."""
    response = client.chat.completions.create(
        model=MODELO,
        messages=[
            {"role": "system", "content": "Eres un asistente útil."},
            {"role": "user", "content": message},
        ],
        temperature=1.0,
        top_p=0.95,
    )
    return response.choices[0].message.content

In [10]:
chatbot = gr.Interface(
    fn=get_response,
    inputs=gr.Textbox(placeholder="Escribe tu pregunta aquí..."),
    outputs="text",
    title="Chatbot con NVIDIA NIM",
    description="Conversa con NVIDIA Nemotron 3 Super.",
    flagging_mode="never",
)

In [11]:
# Lanzamos el chatbot
chatbot.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://9c2cde7b44a24784bb.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
chatbot.launch(
    server_name="0.0.0.0",
    server_port=8081,
    show_error=True
)

In [12]:
chatbot.close()

Closing server running on port: 7860


### Análisis — Chatbot básico

`get_response` construye una solicitud con un mensaje `system` que determina el comportamiento general del asistente y un mensaje `user` con la pregunta. Usa `client.chat.completions.create` para consultar NVIDIA NIM y devuelve la respuesta completa con `return`. El ejemplo **no guarda las interacciones previas**: cada invocación incluye únicamente el prompt del sistema y la pregunta actual.

**Prueba sugerida:** preguntar «¿Qué es una red neuronal?» y después «¿Qué ejemplo me diste?». El segundo mensaje no se envía con el contexto del primero, por lo que no debe asumirse que el modelo recuerda la conversación.

## 4. Chatbot con Streaming

El streaming muestra la respuesta a medida que se genera, token por token.
En Gradio esto se logra con `yield` en lugar de `return` — la función se convierte
en un generador que devuelve la respuesta parcial en cada paso.

In [13]:
def stream_response(message: str):
    """Genera una respuesta de NVIDIA NIM en modo streaming."""
    respuesta = ""

    stream = client.chat.completions.create(
        model=MODELO,
        messages=[
            {"role": "system", "content": "Eres un asistente útil."},
            {"role": "user", "content": message},
        ],
        temperature=1.0,
        top_p=0.95,
        stream=True,
    )

    for chunk in stream:
        texto = chunk.choices[0].delta.content if chunk.choices else None
        if texto:
            respuesta += texto
            yield respuesta

In [14]:
# Gradio detecta automáticamente que la función es un generador (usa yield)
# y activa el modo streaming en la interfaz
streaming_chatbot = gr.Interface(
    fn=stream_response,
    inputs="textbox",
    outputs="text",
    title="Chatbot con Streaming",
    description="Las respuestas aparecen en tiempo real a medida que se generan",
    flagging_mode="never"
)

In [15]:
streaming_chatbot.launch(
    server_name="0.0.0.0",
    server_port=8081,
    show_error=True
)

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://0df1a96cac16ca9ef9.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [19]:
streaming_chatbot.close()

Closing server running on port: 8081


### Análisis — Streaming

En esta versión se activa `stream=True`. La API entrega fragmentos de respuesta (`chunk`) y el generador concatena el contenido disponible en la variable `respuesta`. `yield respuesta` actualiza la interfaz de forma progresiva. En comparación con el chatbot básico, el usuario puede empezar a leer antes de que termine toda la generación; esto **no garantiza** que el tiempo total de respuesta sea menor.

**Prueba sugerida:** pedir una explicación de varios párrafos y observar cómo se va completando el texto.

## 5. Chatbot con memoria

`gr.ChatInterface` está diseñado para chatbots con historial. Gradio gestiona la
interfaz y entrega a nuestra función el mensaje actual y los intercambios
anteriores. Nosotros convertimos ese historial al formato estándar de mensajes
de la API.

In [20]:
def get_text(content):
    """Extrae texto de los formatos de contenido usados por Gradio."""
    if isinstance(content, str):
        return content
    if isinstance(content, list) and content:
        first = content[0]
        return first.get("text", "") if isinstance(first, dict) else str(first)
    return str(content)


def history_to_messages(history: list) -> list:
    """Convierte historiales nuevos o antiguos de Gradio a mensajes OpenAI."""
    messages = []
    for entry in history:
        if isinstance(entry, dict):
            messages.append({
                "role": entry["role"],
                "content": get_text(entry["content"]),
            })
        elif isinstance(entry, (list, tuple)) and len(entry) == 2:
            user_msg, assistant_msg = entry
            messages.append({"role": "user", "content": get_text(user_msg)})
            if assistant_msg:
                messages.append({
                    "role": "assistant",
                    "content": get_text(assistant_msg),
                })
    return messages


def chat_con_memoria(message: str, history: list):
    messages = [{"role": "system", "content": "Eres un asistente útil."}]
    messages.extend(history_to_messages(history))
    messages.append({"role": "user", "content": message})

    respuesta = ""
    stream = client.chat.completions.create(
        model=MODELO,
        messages=messages,
        temperature=1.0,
        top_p=0.95,
        stream=True,
    )
    for chunk in stream:
        texto = chunk.choices[0].delta.content if chunk.choices else None
        if texto:
            respuesta += texto
            yield respuesta

In [21]:
# gr.ChatInterface se encarga de toda la UI del chat:
# burbuja de mensajes, historial visual, campo de entrada, etc.
memory_chatbot = gr.ChatInterface(
    fn=chat_con_memoria,
    title="Chatbot con Memoria",
    description="El asistente recuerda el contexto de toda la conversacion.",
    examples=[
        "Cuentame sobre el aprendizaje automatico",
        "Como funcionan las redes neuronales?",
        "Cual es la diferencia entre IA y ML?"
    ],
    flagging_mode="never"
)

In [22]:
memory_chatbot.launch(
    server_name="0.0.0.0",
    server_port=8081,
    show_error=True
)

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://b15fc3cc09862a2d0e.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [23]:
memory_chatbot.close()

Closing server running on port: 8081


### Análisis — Memoria conversacional

`gr.ChatInterface` recibe el mensaje actual y el historial de conversación. La función `history_to_messages` transforma los turnos anteriores en mensajes con roles compatibles con la API. Después se añade la nueva pregunta y se envía todo el conjunto al modelo. Por eso la memoria es **contexto reenviado en cada petición**, no un almacenamiento permanente ni un nuevo entrenamiento del modelo. Su tamaño puede aumentar el consumo de tokens.

**Prueba sugerida:** escribir «Mi color favorito es azul» y, en el mismo chat, preguntar «¿Cuál es mi color favorito?». Comparar con la versión sin memoria.

## 6. Generador de Menús de Restaurante

Veamos un ejemplo de aplicación con múltiples entradas. Gradio puede manejar
formularios complejos — aquí el usuario proporciona tres campos y obtiene
un menú generado por IA en formato Markdown.

In [ ]:
def generate_menu(nombre_restaurante: str, tipo_cocina: str, requisitos_especiales: str = "Ninguno") -> str:
    """Genera un menú de restaurante usando NVIDIA NIM."""
    prompt = f"""
    Crea un menú para "{nombre_restaurante}", un restaurante de cocina {tipo_cocina}.
    Requisitos especiales: {requisitos_especiales}

    Incluye 3 entradas, 4 platos principales y 2 postres. Para cada elemento
    incluye nombre, descripción breve y precio en pesos colombianos. Formatea
    la respuesta en Markdown y agrega una introducción corta.
    """

    response = client.chat.completions.create(
        model=MODELO,
        messages=[
            {
                "role": "system",
                "content": "Eres un consultor experto en restaurantes y diseño de menús.",
            },
            {"role": "user", "content": prompt},
        ],
        temperature=1.0,
        top_p=0.95,
    )
    return response.choices[0].message.content

In [ ]:
# Interfaz con múltiples inputs y output en Markdown
menu_generator = gr.Interface(
    fn=generate_menu,
    inputs=[
        gr.Textbox(label="Nombre del Restaurante"),
        gr.Textbox(label="Tipo de Cocina (ej. italiana, japonesa, colombiana)"),
        gr.Textbox(
            label="Requisitos Especiales (Opcional)",
            placeholder="ej. opciones vegetarianas, sin gluten"
        )
    ],
    outputs=gr.Markdown(label="Menu Generado"),
    title="Generador de Menus con IA",
    description="Crea un menu profesional para tu restaurante en segundos",
    flagging_mode="never"
)

In [ ]:
menu_generator.launch(
    server_name="0.0.0.0",
    server_port=8081,
    show_error=True
)

In [ ]:
menu_generator.close()

### Análisis — Generador de menús

Esta interfaz muestra que Gradio admite varias entradas en una sola aplicación. El nombre del restaurante, tipo de cocina y requisitos especiales se integran en un prompt estructurado. Se solicita una salida en Markdown con tres entradas, cuatro platos principales y dos postres, incluyendo precios en pesos colombianos. Los precios son **propuestas generadas por el modelo**, no cotizaciones ni información de mercado validada.

**Prueba sugerida:** generar un menú para «Sabores del Valle», cocina colombiana, con opciones vegetarianas; verificar el número de platos y que se cumplan los requisitos.

## 7. Personalizando el System Prompt

Una extensión útil: permitir al usuario definir el comportamiento del asistente
cambiando el system prompt desde la interfaz.

In [ ]:
def chat_personalizable(message: str, history: list, system_prompt: str):
    """Chat en el que el usuario personaliza el system prompt."""
    system_prompt = system_prompt or "Eres un asistente útil."

    messages = [{"role": "system", "content": system_prompt}]
    messages.extend(history_to_messages(history))
    messages.append({"role": "user", "content": message})

    respuesta = ""
    stream = client.chat.completions.create(
        model=MODELO,
        messages=messages,
        temperature=1.0,
        top_p=0.95,
        stream=True,
    )
    for chunk in stream:
        texto = chunk.choices[0].delta.content if chunk.choices else None
        if texto:
            respuesta += texto
            yield respuesta

In [ ]:
# Usamos gr.ChatInterface con additional_inputs para agregar el campo de system prompt
custom_chatbot = gr.ChatInterface(
    fn=chat_personalizable,
    title="Chatbot Personalizable",
    description="Define como quieres que se comporte el asistente.",
    additional_inputs=[
        gr.Textbox(
            label="System Prompt",
            placeholder="ej. Eres un experto en Python que responde solo con codigo",
            value="Eres un asistente util y conciso."
        )
    ],
    examples=[
        ["Explica que es una red neuronal", "Eres un profesor universitario."],
        ["Escribe un haiku sobre la programacion", "Eres un poeta."],
    ],
    flagging_mode="never"
)

In [ ]:
custom_chatbot.launch(
    server_name="0.0.0.0",
    server_port=8081,
    show_error=True
)

In [ ]:
custom_chatbot.close()

### Análisis — System prompt personalizable

`additional_inputs` agrega un campo donde el usuario decide las instrucciones generales del chatbot. La función `chat_personalizable` coloca estas instrucciones en el mensaje `system`, conserva el historial y genera la respuesta con streaming. Así se puede observar cómo el rol y las instrucciones del sistema afectan la forma y el contenido de las respuestas; no obstante, el cumplimiento no está garantizado en todos los casos.

**Prueba sugerida:** utilizar «Responde en una sola oración y con lenguaje sencillo» y solicitar «Explica qué es el aprendizaje automático». Cambiar el prompt por «Responde como profesor universitario con un ejemplo» y comparar el estilo.

## 8. Comparación y conclusiones

| Implementación | ¿Usa NVIDIA NIM? | ¿Tiene memoria del chat? | ¿Utiliza streaming? | Objetivo |
|---|---|---|---|---|
| Echo Bot | No | No | No | Comprender las entradas y salidas de Gradio |
| Chatbot básico | Sí | No | No | Integrar una API de LLM en una interfaz |
| Chatbot con streaming | Sí | No | Sí | Mostrar la respuesta de forma progresiva |
| Chatbot con memoria | Sí | Sí, mediante historial enviado | Sí | Mantener el contexto entre mensajes |
| Generador de menús | Sí | No | No | Trabajar con varias entradas y salida Markdown |
| Chatbot personalizable | Sí | Sí, mediante historial enviado | Sí | Adaptar el comportamiento con un system prompt |

### Conclusiones

1. **Gradio simplifica la creación de interfaces web** porque enlaza funciones de Python con componentes visuales sin necesidad de construir un frontend desde cero.
2. **NVIDIA NIM proporciona el modelo mediante una API**, por lo cual no es necesario descargar sus pesos ni ejecutarlo en el equipo para este laboratorio. Sí hace falta acceso a la API y una clave válida.
3. **Streaming y memoria son mecanismos diferentes:** streaming mejora la presentación progresiva de la respuesta, mientras que la memoria añade turnos anteriores a cada solicitud.
4. **Los prompts de sistema permiten personalizar el asistente** y formularios como el generador de menús ejemplifican aplicaciones más específicas que un chat genérico.
5. **Limitaciones:** las respuestas pueden contener errores; el historial incrementa el contexto enviado y las llamadas remotas dependen de la conexión, los límites y la disponibilidad del servicio.

### Estado de la práctica

El notebook original contiene código para todas las secciones y algunas salidas de ejecución guardadas. Estos análisis describen lo que implementa el código; **no representan una nueva ejecución de la API ni una validación actual del modelo**. Para comprobarlo, configura tu clave, ejecuta las celdas en orden y realiza las pruebas propuestas.
